In [3]:
from pathlib import Path
import json
import shutil
from collections import defaultdict

# Project paths
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "trashcan_cleaned_final"

YOLO_PATH = PROJECT_ROOT / "data" / "processed" / "trashcan_yolo"

ANNOTATIONS_PATH = DATASET_PATH / "annotations"

# Create output folders
for split in ["train", "val"]:
    (YOLO_PATH / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_PATH / "labels" / split).mkdir(parents=True, exist_ok=True)

# Load COCO JSON
with open(ANNOTATIONS_PATH / "instances_train.json", "r", encoding="utf-8") as f:
    train_coco = json.load(f)

with open(ANNOTATIONS_PATH / "instances_val.json", "r", encoding="utf-8") as f:
    val_coco = json.load(f)

# Map COCO category IDs to YOLO IDs
categories = sorted(train_coco["categories"], key=lambda c: c["id"])

category_mapping = {
    category["id"]: index
    for index, category in enumerate(categories)
}

class_names = [category["name"] for category in categories]

print("YOLO class mapping:")
for class_id, class_name in enumerate(class_names):
    print(class_id, ":", class_name)


def convert_split(coco, split_name):

    source_images = DATASET_PATH / split_name / "images"

    annotations_by_image = defaultdict(list)

    for ann in coco["annotations"]:
        annotations_by_image[ann["image_id"]].append(ann)

    converted_annotations = 0
    negative_images = 0

    file_names = [img["file_name"] for img in coco["images"]]

    if len(file_names) != len(set(file_names)):
        raise ValueError(f"Duplicate filenames in {split_name}")

    for image_info in coco["images"]:

        image_id = image_info["id"]
        image_name = Path(image_info["file_name"])

        source_path = source_images / image_name
        destination_image = YOLO_PATH / "images" / split_name / image_name

        if not source_path.exists():
            raise FileNotFoundError(source_path)

        destination_image.parent.mkdir(parents=True, exist_ok=True)

        shutil.copy2(source_path, destination_image)

        label_path = (
            YOLO_PATH / "labels" / split_name /
            image_name.with_suffix(".txt")
        )

        label_path.parent.mkdir(parents=True, exist_ok=True)

        width = image_info["width"]
        height = image_info["height"]

        image_annotations = annotations_by_image.get(image_id, [])

        if len(image_annotations) == 0:
            negative_images += 1

        lines = []

        for ann in image_annotations:

            x, y, box_width, box_height = ann["bbox"]

            if box_width <= 0 or box_height <= 0:
                raise ValueError(f"Invalid bbox: {ann['id']}")

            if (
                x < 0 or y < 0 or
                x + box_width > width + 0.01 or
                y + box_height > height + 0.01
            ):
                raise ValueError(f"Bbox outside image: {ann['id']}")

            class_id = category_mapping[ann["category_id"]]

            x_center = (x + box_width / 2) / width
            y_center = (y + box_height / 2) / height

            normalized_width = box_width / width
            normalized_height = box_height / height

            line = (
                f"{class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{normalized_width:.6f} "
                f"{normalized_height:.6f}"
            )

            lines.append(line)

            converted_annotations += 1

        # Empty file for negative images
        label_path.write_text(
            "\n".join(lines),
            encoding="utf-8"
        )

    print(f"\n{split_name.upper()} CONVERSION")
    print("Images:", len(coco["images"]))
    print("Annotations converted:", converted_annotations)
    print("Negative images:", negative_images)


convert_split(train_coco, "train")
convert_split(val_coco, "val")


# Create YOLO configuration file
yaml_content = f"""path: {YOLO_PATH.as_posix()}
train: images/train
val: images/val

names:
"""

for index, name in enumerate(class_names):
    yaml_content += f"  {index}: {name}\n"

(YOLO_PATH / "data.yaml").write_text(
    yaml_content,
    encoding="utf-8"
)

print("\nYOLO dataset created successfully!")
print("Dataset path:", YOLO_PATH)
print("Configuration:", YOLO_PATH / "data.yaml")

YOLO class mapping:
0 : trash_etc
1 : trash_plastic
2 : trash_metal
3 : trash_wood
4 : trash_fabric
5 : trash_fishing_gear
6 : trash_paper
7 : trash_rubber

TRAIN CONVERSION
Images: 5233
Annotations converted: 4942
Negative images: 1058

VAL CONVERSION
Images: 1249
Annotations converted: 1272
Negative images: 202

YOLO dataset created successfully!
Dataset path: c:\Users\mariem\Documents\trashcan-dl-mlops\data\processed\trashcan_yolo
Configuration: c:\Users\mariem\Documents\trashcan-dl-mlops\data\processed\trashcan_yolo\data.yaml


In [4]:
from pathlib import Path
from collections import Counter

YOLO_PATH = PROJECT_ROOT / "data" / "processed" / "trashcan_yolo"

def validate_yolo_split(split_name):

    images_path = YOLO_PATH / "images" / split_name
    labels_path = YOLO_PATH / "labels" / split_name

    image_files = [
        p for p in images_path.iterdir()
        if p.is_file()
    ]

    label_files = list(labels_path.glob("*.txt"))

    errors = []
    total_annotations = 0
    empty_labels = 0
    class_counts = Counter()

    image_stems = [p.stem for p in image_files]

    if len(image_stems) != len(set(image_stems)):
        errors.append("Duplicate image filenames (same stem)")

    image_stem_set = set(image_stems)
    label_stem_set = {p.stem for p in label_files}

    missing_labels = image_stem_set - label_stem_set
    extra_labels = label_stem_set - image_stem_set

    if missing_labels:
        errors.append(f"{len(missing_labels)} images without labels")

    if extra_labels:
        errors.append(f"{len(extra_labels)} labels without images")

    for label_file in label_files:

        content = label_file.read_text(
            encoding="utf-8"
        ).strip()

        if not content:
            empty_labels += 1
            continue

        for line_number, line in enumerate(content.splitlines(), start=1):

            values = line.split()

            if len(values) != 5:
                errors.append(
                    f"{label_file.name}, line {line_number}: "
                    "expected 5 values"
                )
                continue

            try:
                class_id = int(values[0])
                x_center, y_center, width, height = map(
                    float, values[1:]
                )
            except ValueError:
                errors.append(
                    f"{label_file.name}, line {line_number}: "
                    "invalid numeric values"
                )
                continue

            if class_id < 0 or class_id >= 8:
                errors.append(f"Invalid class ID: {class_id}")

            if not all(
                0 <= value <= 1
                for value in [x_center, y_center, width, height]
            ):
                errors.append(
                    f"Invalid normalized coordinates: {label_file.name}"
                )

            if width <= 0 or height <= 0:
                errors.append(f"Invalid box dimensions: {label_file.name}")

            total_annotations += 1
            class_counts[class_id] += 1

    print(f"\n--- {split_name.upper()} YOLO VALIDATION ---")
    print("Images:", len(image_files))
    print("Label files:", len(label_files))
    print("Annotations:", total_annotations)
    print("Empty label files:", empty_labels)
    print("Class distribution:", dict(sorted(class_counts.items())))
    print("Errors:", len(errors))

    if errors:
        for error in errors[:15]:
            print("-", error)
    else:
        print("All checks passed!")


validate_yolo_split("train")
validate_yolo_split("val")

print("\n--- DATA.YAML ---")
print((YOLO_PATH / "data.yaml").read_text(encoding="utf-8"))


--- TRAIN YOLO VALIDATION ---
Images: 5233
Label files: 5233
Annotations: 4942
Empty label files: 1058
Class distribution: {0: 1589, 1: 1467, 2: 908, 3: 285, 4: 258, 5: 160, 6: 168, 7: 107}
Errors: 0
All checks passed!

--- VAL YOLO VALIDATION ---
Images: 1249
Label files: 1249
Annotations: 1272
Empty label files: 202
Class distribution: {0: 450, 1: 397, 2: 218, 3: 55, 4: 56, 5: 36, 6: 25, 7: 35}
Errors: 0
All checks passed!

--- DATA.YAML ---
path: c:/Users/mariem/Documents/trashcan-dl-mlops/data/processed/trashcan_yolo
train: images/train
val: images/val

names:
  0: trash_etc
  1: trash_plastic
  2: trash_metal
  3: trash_wood
  4: trash_fabric
  5: trash_fishing_gear
  6: trash_paper
  7: trash_rubber

